# sample 20 lemmas to run 5 times

In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
import typing as t
from pprint import pprint
import pickle
import random
import json
import csv

from src.config import CONFIG
from src.dataset.pnvrocqlib import PNV_ROCQLIB_RETRYING_SAMPLED_FILE, PNV_ROCQLIB_RETRYING_SAMPLED_CSV_FILE, PNV_ROCQLIB_TEST_DATASET

len(PNV_ROCQLIB_TEST_DATASET)

In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
run_directory = root_directory / "data/evaluation/goal_decomposition/pnvrocqlib_test_with_hammer_max_depth_5_preceding-lemmas-only-ec66921417dc47bebac0cdac7362c78e"

example_result_files = list(run_directory.glob("*.json"))
example_result_files = [file for file in example_result_files if any(example.name == file.stem for example in PNV_ROCQLIB_TEST_DATASET)]
len(example_result_files)

In [ ]:
example_results = {
    file.stem: json.load(open(file))
    for file in example_result_files
}

example_root_nodes = {
    filename: example_result["nodes"][0]
    for filename, example_result in example_results.items()
}

assert all(example_root_node["parent_uuid"] is None for example_root_node in example_root_nodes.values())

example_proofs = {
    filename: example_result["proof"]
    for filename, example_result in example_root_nodes.items()
}

print("has_proof", len([proof for proof in example_proofs.values() if proof is not None]))
print("no_proof", len([proof for proof in example_proofs.values() if proof is None]))


In [ ]:
examples_without_proofs = [
    filename for filename, proof in example_proofs.items() if proof is None
]

examples_with_bullets = [
    filename for filename, proof in example_proofs.items() if proof is not None and "\n- " in proof
]

sampled_examples = [
    *examples_with_bullets,
    *random.sample(examples_without_proofs, min(len(examples_without_proofs), 10)),
]

sampled_examples




In [ ]:
dataset = [
    example for example in PNV_ROCQLIB_TEST_DATASET if example.name in sampled_examples
]

len(dataset)





In [ ]:
if not PNV_ROCQLIB_RETRYING_SAMPLED_FILE.exists():
    with PNV_ROCQLIB_RETRYING_SAMPLED_FILE.open("wb") as f:
        pickle.dump(dataset, f)
    TEST_EXAMPLE_NAMES = [example.name for example in dataset]
    with PNV_ROCQLIB_RETRYING_SAMPLED_CSV_FILE.open("w") as f:
        writer = csv.writer(f)
        writer.writerow(["example_name"])
        for example_name in TEST_EXAMPLE_NAMES:
            writer.writerow([example_name])